# 04. Cabbage test windows — all cohorts

Check **fixed May–August** and **green-up-aligned four-month** windows across all dedicated test campaigns: primary JECAM, non-JECAM CropHarvest, EuroCrops cabbage, and any additional saved test policies. Only cabbage enters the counts. EuroCrops is a same-source parcel holdout because it also supplies development carrots.

This local diagnostic does not train a model, collect imagery, or replace primary JECAM preprocessing inputs. Collection targets are not accuracy evidence; inferred green-up dates are not verified crop dates.


## 1. Setup and configuration

Choose the project Python 3.12+ environment and **Restart Kernel → Run All**. Finish development collection first so its footprints can be screened. Completed test batches can be assessed while other test batches are pending; those pending batches remain explicitly unassessed. If collection metadata changes during the check, rerun after collection settles.


In [1]:
from pathlib import Path
import json
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root / "src"))
import pandas as pd
from IPython.display import display
from crop_pipeline import cabbage_test_check
from crop_pipeline.preprocessing_workflow import paths

_, report_root, _ = paths(root)
print("Collection report root:", report_root)


Collection report root: ./sen4agrinet_work/inventory


## 2. Inspect all test cohorts

Discover test campaigns and verify completed handoff hashes. Missing campaigns and pending batches are shown, rather than interpreted as failed cabbage windows. California carrot and carrot/potato-only campaigns normally contribute no cabbage.


In [2]:
inventory = cabbage_test_check.discover(report_root)
display(pd.DataFrame([{
    "cohort": c["cohort"], "campaign": c["campaign"], "status": c["status"],
    "completed_handoffs": len(c["sources"]),
    "unassessed_batches": sum(b["status"] != "completed" or not b["has_receipt"] for b in c["batches"]),
} for c in inventory]))
display(pd.DataFrame([{ "cohort": c["cohort"], **b } for c in inventory for b in c["batches"]]))


,cohort,campaign,status,completed_handoffs,unassessed_batches
0,jecam,9803c314c949,completed_inputs_available,19,5
1,non_jecam_cabbage_supplement_v1,223c7df494ac,completed_inputs_available,1,0
2,non_jecam_carrot_potato_supplement_v1,984f51124874,completed_inputs_available,6,0
3,california_dwr_carrot_2023_v1,a3ae04a8ea2f,completed_inputs_available,2,0
4,eurocrops_nl_cabbage_2021_v1,ba5a6dfb78cd,completed_inputs_available,9,0


,cohort,batch,status,has_receipt
0,jecam,...,completed,True
1,jecam,...,completed,True
2,jecam,...,completed,True
3,jecam,...,completed,True
4,jecam,...,completed,True
5,jecam,...,completed,True
6,jecam,...,skipped,False
7,jecam,...,skipped,False
8,jecam,...,completed,True
9,jecam,...,completed,True


## 3. Check imagery and select windows

Use the existing quality masks, **80% validity**, **eight complete half-month bins**, finite feature checks, timing qualification, and **20-pixel parcel cap**. Exclude cabbage footprints overlapping development or earlier test cohorts, with JECAM taking precedence. Fixed windows run May 1 through September 1 (exclusive); aligned windows follow the existing green-up selection rules. Test parcels are excluded from threshold-sensitivity experiments. Each run writes a new diagnostic folder and leaves collection state and primary preprocessing pointers unchanged.


In [3]:
check_folder = cabbage_test_check.run(root)
print("Completed cabbage test check:", check_folder)


Regions audited: 211
Composite outcomes: {'low_coverage': 581, 'quality_pass': 4483}
Composite timeline rows: 5064
Detector outcomes: {'ambiguous': 40, 'candidate': 34, 'rejected': 18}
Results: ./sen4agrinet_work/analysis/cabbage_test_checks/check_8riz861r/9803c314c949/greenup
Reference-supported field-seasons: 0
Initial estimated proposals: 43
Automatically qualified estimates: 31
Total qualified estimated windows: 31
Unresolved outcomes: 61
Automatic timing failures: 12
Review templates: ./sen4agrinet_work/analysis/cabbage_test_checks/check_8riz861r/9803c314c949/seasons/review
Reference-supported windows saved: 0
Estimated observation windows saved: 31
Audit: ./sen4agrinet_work/analysis/cabbage_test_checks/check_8riz861r/9803c314c949/seasons/season_review_audit.csv
Cycle selection: {'selected': 12, 'excluded_later': 4, 'excluded_incomplete': 15}
Selected estimated windows: 12
Regions audited: 37
Composite outcomes: {'quality_pass': 711, 'low_coverage': 177}
Composite timeline rows: 8

## 4. Compare fixed and green-up coverage

`aligned` means green-up-aligned. Full populations use each recipe's qualifying pixels; matched populations use identical locations in both recipes. Compare them separately; do not add their counts. The combined row excludes overlaps and deduplicates parcel locations. Targets are **20 parcels / 400 pixels** for cabbage testing; cohort-level shortfalls are descriptive, not extra required quotas. Missing or pending imagery is not assessed.


In [4]:
counts = pd.read_csv(check_folder / "window_counts.csv")
display(counts)
combined = counts[counts["test_cohort"] == "all_test_cohorts"]
display(combined)


,test_cohort,window,population,usable_parcels,usable_pixels,parcel_shortfall,pixel_shortfall,target_met,campaign
0,jecam,fixed,full,38,351,0,49,False,9803c314c949
1,jecam,fixed,matched,11,93,9,307,False,9803c314c949
2,jecam,aligned,full,12,95,8,305,False,9803c314c949
3,jecam,aligned,matched,11,93,9,307,False,9803c314c949
4,non_jecam_cabbage_supplement_v1,fixed,full,12,113,8,287,False,223c7df494ac
5,non_jecam_cabbage_supplement_v1,fixed,matched,5,48,15,352,False,223c7df494ac
6,non_jecam_cabbage_supplement_v1,aligned,full,8,89,12,311,False,223c7df494ac
7,non_jecam_cabbage_supplement_v1,aligned,matched,5,48,15,352,False,223c7df494ac
8,non_jecam_carrot_potato_supplement_v1,fixed,full,0,0,20,400,False,984f51124874
9,non_jecam_carrot_potato_supplement_v1,fixed,matched,0,0,20,400,False,984f51124874


,test_cohort,window,population,usable_parcels,usable_pixels,parcel_shortfall,pixel_shortfall,target_met,campaign
20,all_test_cohorts,fixed,full,59,634,0,0,True,all_test_cohorts
21,all_test_cohorts,fixed,matched,16,141,4,259,False,all_test_cohorts
22,all_test_cohorts,aligned,full,20,184,0,216,False,all_test_cohorts
23,all_test_cohorts,aligned,matched,16,141,4,259,False,all_test_cohorts


## 5. Inspect parcel outcomes and timing diagnostics

Zero usable pixels can reflect overlap exclusion, missing/poor imagery, incomplete windows, or timing rejection. Inspect the cohort's `quality/`, `fixed_losses.csv`, `greenup/candidate_greenup.csv`, and `seasons/automatic_timing_audit.csv` for the reason. The notebook keeps per-parcel outcomes, including parcels with no qualifying window.


In [5]:
parcel_path = check_folder / "parcel_outcomes.csv"
if parcel_path.exists() and parcel_path.stat().st_size > 1:
    display(pd.read_csv(parcel_path))
else:
    print("No completed cabbage parcels were available to assess.")
for diagnostic in ("fixed_losses.csv", "greenup/candidate_greenup.csv", "seasons/automatic_timing_audit.csv"):
    for path in sorted(check_folder.glob("*/" + diagnostic)):
        print(path)


,campaign,test_cohort,parcel_group,overlap_excluded,fixed_pixels,aligned_pixels
0,9803c314c949,jecam,cropharvest|jecam|2019|jecam:6891,False,2,2
1,9803c314c949,jecam,cropharvest|jecam|2021|jecam:12192,False,4,0
2,9803c314c949,jecam,cropharvest|jecam|2021|jecam:12341,False,0,0
3,9803c314c949,jecam,cropharvest|jecam|2021|jecam:12367,False,9,0
4,9803c314c949,jecam,cropharvest|jecam|2021|jecam:12379,False,12,12
...,...,...,...,...,...,...
394,ba5a6dfb78cd,eurocrops_nl_cabbage_2021_v1,eurocrops|NL|2021|685550,False,0,0
395,ba5a6dfb78cd,eurocrops_nl_cabbage_2021_v1,eurocrops|NL|2021|695007,False,0,0
396,ba5a6dfb78cd,eurocrops_nl_cabbage_2021_v1,eurocrops|NL|2021|720615,False,0,0
397,ba5a6dfb78cd,eurocrops_nl_cabbage_2021_v1,eurocrops|NL|2021|738852,False,0,0


./sen4agrinet_work/analysis/cabbage_test_checks/check_8riz861r/223c7df494ac/fixed_losses.csv
./sen4agrinet_work/analysis/cabbage_test_checks/check_8riz861r/9803c314c949/fixed_losses.csv
./sen4agrinet_work/analysis/cabbage_test_checks/check_8riz861r/ba5a6dfb78cd/fixed_losses.csv
./sen4agrinet_work/analysis/cabbage_test_checks/check_8riz861r/223c7df494ac/greenup/candidate_greenup.csv
./sen4agrinet_work/analysis/cabbage_test_checks/check_8riz861r/9803c314c949/greenup/candidate_greenup.csv
./sen4agrinet_work/analysis/cabbage_test_checks/check_8riz861r/ba5a6dfb78cd/greenup/candidate_greenup.csv
./sen4agrinet_work/analysis/cabbage_test_checks/check_8riz861r/223c7df494ac/seasons/automatic_timing_audit.csv
./sen4agrinet_work/analysis/cabbage_test_checks/check_8riz861r/9803c314c949/seasons/automatic_timing_audit.csv
./sen4agrinet_work/analysis/cabbage_test_checks/check_8riz861r/ba5a6dfb78cd/seasons/automatic_timing_audit.csv


## 6. Summary and next action

Save the counts with the collection inventory and receipt. Rerun after pending imagery is collected. Assess JECAM independently when reporting results, and label the broader combined population explicitly. This check does not enroll supplements in the standard model evaluation.


In [6]:
receipt = json.loads((check_folder / "check_receipt.json").read_text())
print("Status:", receipt["status"])
print("Scope:", receipt["scope"])
print("Counts:", check_folder / "window_counts.csv")
print("Parcel outcomes:", check_folder / "parcel_outcomes.csv")
print("Inventory:", check_folder / "collection_inventory.json")
print("Receipt:", check_folder / "check_receipt.json")


Status: completed
Scope: Completed local cabbage test imagery only; pending batches are unassessed
Counts: ./sen4agrinet_work/analysis/cabbage_test_checks/check_8riz861r/window_counts.csv
Parcel outcomes: ./sen4agrinet_work/analysis/cabbage_test_checks/check_8riz861r/parcel_outcomes.csv
Inventory: ./sen4agrinet_work/analysis/cabbage_test_checks/check_8riz861r/collection_inventory.json
Receipt: ./sen4agrinet_work/analysis/cabbage_test_checks/check_8riz861r/check_receipt.json


## Evidence and interpretation

Coverage describes this saved snapshot. Green-up is estimated timing; sample targets do not establish accuracy. See [research basis and limits](../../docs/research_basis.md) for sources and project-specific choices.
